# [6029.09 LB] NeuroGolf All-Task ONNX Solution

This notebook releases my current full-submission ONNX bundle for **The 2026 NeuroGolf Championship**.

At the time of release, the highest clearly labeled public-code baseline I found was **5800.55 LB** (`NeuroGolf Task-Level ONNX Blend`). This bundle raises the public score to **6029.09**.

The main point of this release is practical: every task now has a working ONNX file in the attached `submission.zip`. Roughly speaking, the community no longer needs to spend most effort searching for any valid solver for each task; the next high-leverage direction is reducing the cost of these task solvers while preserving correctness.

## What is included

- Kaggle Dataset attachment: 400 ONNX files, `task001.onnx` through `task400.onnx`; the notebook repacks them into `submission.zip`.
- Public LB score: **6029.09**.
- Manifest SHA256 of the 400 ONNX file contents: `36b31377709b6fde4356f84ba5c2c7a6a8c2283f7e3fb29d99d6818ebea12ad2`.

This is a release of the latest complete bundle I have been using. Some components build on public work and blending ideas; several tasks were repaired with additional rule-specific ONNX solvers. The goal here is to make the current higher-scoring state reproducible and easy to build on.

In [1]:
from pathlib import Path
import hashlib
import zipfile

EXPECTED_MANIFEST_SHA256 = "36b31377709b6fde4356f84ba5c2c7a6a8c2283f7e3fb29d99d6818ebea12ad2"

src_dir = Path("/kaggle/input/neurogolf-6029-submission-bundle")
if not src_dir.exists():
    raise FileNotFoundError(
        "Expected Kaggle dataset not found: " + str(src_dir) + "; "
        "please add dataset jsrdcht/neurogolf-6029-submission-bundle to this notebook."
    )

onnx_files = sorted(src_dir.glob("task*.onnx"))
print("source dataset:", src_dir)
print("number of ONNX files:", len(onnx_files))
print("first files:", [p.name for p in onnx_files[:3]])
print("last files:", [p.name for p in onnx_files[-3:]])
assert len(onnx_files) == 400
assert onnx_files[0].name == "task001.onnx"
assert onnx_files[-1].name == "task400.onnx"

manifest_lines = []
for p in onnx_files:
    data = p.read_bytes()
    manifest_lines.append(f"{p.name}\t{len(data)}\t{hashlib.sha256(data).hexdigest()}")
manifest = "\n".join(manifest_lines) + "\n"
manifest_sha = hashlib.sha256(manifest.encode()).hexdigest()
print("manifest_sha256:", manifest_sha)
assert manifest_sha == EXPECTED_MANIFEST_SHA256

out = Path("/kaggle/working/submission.zip") if Path("/kaggle/working").exists() else Path("submission.zip")
with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in onnx_files:
        z.write(p, arcname=p.name)

with zipfile.ZipFile(out) as z:
    names = sorted(z.namelist())
print("created submission:", out)
print("zip entries:", len(names), names[:3], names[-3:])
assert len(names) == 400
assert names[0] == "task001.onnx"
assert names[-1] == "task400.onnx"


source dataset: /kaggle/input/neurogolf-6029-submission-bundle
number of ONNX files: 400
first files: ['task001.onnx', 'task002.onnx', 'task003.onnx']
last files: ['task398.onnx', 'task399.onnx', 'task400.onnx']
manifest_sha256: 36b31377709b6fde4356f84ba5c2c7a6a8c2283f7e3fb29d99d6818ebea12ad2
created submission: /kaggle/working/submission.zip
zip entries: 400 ['task001.onnx', 'task002.onnx', 'task003.onnx'] ['task398.onnx', 'task399.onnx', 'task400.onnx']


## Suggested next steps

This release should be treated as a strong all-task starting point rather than the end of the competition. The most useful follow-up work is now cost reduction:

- replace large ONNX graphs with smaller equivalent logic;
- remove redundant intermediate tensors and constants;
- merge repeated patterns across tasks;
- keep static shapes and avoid leaderboard-fragile tricks;
- verify each replacement task-by-task before blending it back into the full bundle.


## Citation and support

If this release helps your NeuroGolf work, please consider upvoting this notebook and citing or linking back to it in your own public notebooks. This makes it easier for others to trace the source bundle and build on the same baseline.
